# Pi0 Comprehensive Study

This notebook implements three complementary experiments on the Pi0 model:

1. **Baseline Benchmarking**: Record unmodified performance on LIBERO, VLABench, MetaWorld
2. **Ablation Study (Pass0)**: Zero state encoder (`state_proj`) and measure success rate drops
3. **Gradient Analysis**: Measure state encoder gradient contributions using flow matching loss

## Model Details
- **State Encoder**: `state_proj` (Single Linear layer)
- **Loss Function**: Flow matching `F.mse_loss(u_t, v_t)` from Physical-Intelligence/openpi
- **Benchmarks**: LIBERO + VLABench + MetaWorld

## Expected Results
- Baseline provides reference success rates
- Ablation shows performance impact (success rate drop)
- Gradient analysis shows training signal strength (gradient magnitude)
- Cross-validation: Gradient ∝ Performance impact?

---
## Part 1 · Comprehensive Study — lerobot-eval

Uses [lerobot](https://github.com/huggingface/lerobot) for all evaluation.

| Benchmark | Checkpoint | Notes |
|-----------|-----------|-------|
| LIBERO | `lerobot/pi0_libero_finetuned` | Pi0 fine-tuned on LIBERO, has `state_proj` |
| MetaWorld | `lerobot/pi0` | Generalist Pi0, has `state_proj` |
| VLABench | `lerobot/pi0` | Generalist Pi0 + VLABench env |

All baselines via `lerobot-eval` CLI.  
Ablations via Python eval loop with `state_proj` zeroed by a forward hook.

In [ ]:
# ── Paths & workspace ──────────────────────────────────────────────────────
from google.colab import drive, auth as _colab_auth
from pathlib import Path
import os, json, subprocess, sys, torch

drive.mount('/content/drive')
_colab_auth.authenticate_user()       # needed for GCS (VLABench ckpt if used)

WORKSPACE    = '/content/drive/MyDrive/pi0_study'
BASELINE_DIR = Path(f'{WORKSPACE}/Results/baseline')
ABLATION_DIR = Path(f'{WORKSPACE}/Results/ablation')
GRADIENT_DIR = Path(f'{WORKSPACE}/Results/gradient')
LOGS_DIR     = Path('/content/logs')

for d in [BASELINE_DIR, ABLATION_DIR, GRADIENT_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

os.environ['MUJOCO_GL'] = 'egl'
print('✅ Paths ready')


In [ ]:
# ── Install lerobot with LIBERO + MetaWorld extras ─────────────────────────
# Single pip install replaces the entire 4-conda-env + WebSocket server setup
print('Installing lerobot[libero,metaworld] ...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'lerobot[libero,metaworld]'], check=True)

# VLABench environment (for VLABench eval, no WebSocket server needed)
if not Path('/content/VLABench').exists():
    subprocess.run(['git', 'clone', '--quiet',
                    'https://github.com/OpenMOSS/VLABench.git',
                    '/content/VLABench'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    '-e', '/content/VLABench'], check=False)

# Verify state_proj exists in lerobot/pi0
from lerobot.policies.pi0.modeling_pi0 import Pi0Policy
import inspect
assert 'state_proj' in inspect.getsource(Pi0Policy), 'state_proj not found!'
print('✅ lerobot installed — state_proj confirmed in Pi0Policy')


In [ ]:
# ── Shared helpers for policy loading and ablation ─────────────────────────
import numpy as np
from lerobot.policies.pi0.modeling_pi0 import Pi0Policy

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def load_policy(policy_id):
    policy = Pi0Policy.from_pretrained(policy_id)
    return policy.to(DEVICE).eval()

def hook_state_proj(policy):
    """Zero state_proj output. Returns list of hook handles to remove later."""
    handles = []
    for name, module in policy.named_modules():
        if 'state_proj' in name and isinstance(module, torch.nn.Linear):
            h = module.register_forward_hook(lambda m, i, o: torch.zeros_like(o))
            handles.append(h)
            print(f'  ↳ Ablation hook on: {name}')
    if not handles:
        raise RuntimeError('state_proj not found in policy — is this a pi0 (not pi05) model?')
    return handles

def format_obs_libero(obs, device):
    """Convert LIBERO gymnasium obs dict → lerobot batch dict."""
    batch = {}
    if 'agent_pos' in obs:
        batch['observation.state'] = torch.tensor(
            obs['agent_pos'], dtype=torch.float32).unsqueeze(0).to(device)
    if 'pixels' in obs:
        for k, v in obs['pixels'].items():
            img = torch.tensor(v, dtype=torch.float32).permute(2,0,1) / 255.0
            batch[f'observation.images.{k}'] = img.unsqueeze(0).to(device)
    elif 'agentview_image' in obs:
        img = torch.tensor(obs['agentview_image'], dtype=torch.float32).permute(2,0,1) / 255.0
        batch['observation.images.image'] = img.unsqueeze(0).to(device)
    if 'robot0_eye_in_hand_image' in obs:
        img = torch.tensor(obs['robot0_eye_in_hand_image'], dtype=torch.float32).permute(2,0,1) / 255.0
        batch['observation.images.image2'] = img.unsqueeze(0).to(device)
    return batch

def run_episode(policy, env, format_obs_fn, max_steps=500):
    """Run one episode, return success bool."""
    obs, info = env.reset()
    if hasattr(policy, 'reset'):
        policy.reset()
    for _ in range(max_steps):
        batch = format_obs_fn(obs, DEVICE)
        with torch.no_grad():
            action = policy.select_action(batch).squeeze(0).cpu().numpy()
        obs, reward, terminated, truncated, info = env.step(action)
        if info.get('is_success', False) or info.get('success', False):
            return True
        if terminated or truncated:
            break
    return False

print('✅ Helpers ready')


---
## 2. LIBERO Benchmark — `lerobot/pi0_libero_finetuned`

In [ ]:
# ── LIBERO Baseline ────────────────────────────────────────────────────────
# lerobot/pi0_libero_finetuned: Pi0 with state_proj, fine-tuned on LIBERO
# Expected: ~95%+ SR (consistent with Physical Intelligence paper)

LIBERO_SUITES = 'libero_spatial,libero_object,libero_goal,libero_90'
LIBERO_CKPT   = 'lerobot/pi0_libero_finetuned'
LIBERO_EPISODES = 10

print('Running LIBERO baseline...')
result = subprocess.run([
    'lerobot-eval',
    f'--policy.path={LIBERO_CKPT}',
    '--env.type=libero',
    f'--env.task={LIBERO_SUITES}',
    f'--eval.n_episodes={LIBERO_EPISODES}',
    '--eval.batch_size=1',
    '--policy.n_action_steps=10',
    f'--output_dir={BASELINE_DIR}/libero',
], capture_output=True, text=True)

print(result.stdout[-3000:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])


In [ ]:
# ── LIBERO Ablation ────────────────────────────────────────────────────────
# Load pi0_libero_finetuned, zero state_proj, run same eval
# Uses Python eval loop so we can inject the hook before episodes run
import gymnasium as gym

policy = load_policy('lerobot/pi0_libero_finetuned')
handles = hook_state_proj(policy)

LIBERO_SUITE_LIST = ['libero_spatial', 'libero_object', 'libero_goal', 'libero_90']
ablation_results = {}

for suite in LIBERO_SUITE_LIST:
    print(f'\n📍 Ablation: {suite}')
    # lerobot-libero registers environments by suite name
    env = gym.make(f'lerobot/{suite}')
    successes = [run_episode(policy, env, format_obs_libero) for _ in range(LIBERO_EPISODES)]
    env.close()
    sr = sum(successes) / len(successes)
    ablation_results[suite] = {'success_rate': sr, 'successes': successes}
    print(f'  SR (ablated): {sr:.1%}')

for h in handles:
    h.remove()

with open(ABLATION_DIR / 'libero_ablation.json', 'w') as f:
    json.dump(ablation_results, f, indent=2)
print('\n✅ LIBERO ablation done')


---
## 3. MetaWorld Benchmark — `lerobot/pi0` (generalist)

In [ ]:
# ── MetaWorld Baseline ─────────────────────────────────────────────────────
# lerobot/pi0: generalist Pi0, has state_proj
# MetaWorld medium difficulty split (20 tasks)
MW_CKPT     = 'lerobot/pi0'
MW_EPISODES = 20

print('Running MetaWorld baseline...')
result = subprocess.run([
    'lerobot-eval',
    f'--policy.path={MW_CKPT}',
    '--env.type=metaworld',
    '--env.task=medium',
    f'--eval.n_episodes={MW_EPISODES}',
    '--eval.batch_size=1',
    f'--output_dir={BASELINE_DIR}/metaworld',
], capture_output=True, text=True)

print(result.stdout[-3000:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])


In [ ]:
# ── MetaWorld Ablation ─────────────────────────────────────────────────────
import metaworld

policy  = load_policy('lerobot/pi0')
handles = hook_state_proj(policy)

MW_TASKS   = ['reach-v2', 'push-v2', 'pick-place-v2', 'door-open-v2', 'drawer-close-v2']
MW_EPISODES = 10
mw_results  = {}

def format_obs_mw(obs, device):
    """MetaWorld obs is a flat 39-dim state vector + optional pixels."""
    state = torch.tensor(obs if isinstance(obs, np.ndarray) else obs['state'],
                         dtype=torch.float32).unsqueeze(0).to(device)
    return {'observation.state': state}

for task_name in MW_TASKS:
    print(f'\n📍 Ablation: {task_name}')
    ml1  = metaworld.ML1(task_name)
    env  = ml1.train_classes[task_name]()
    task = next(iter(ml1.train_tasks))
    env.set_task(task)
    successes = [run_episode(policy, env, format_obs_mw) for _ in range(MW_EPISODES)]
    env.close()
    sr = sum(successes) / len(successes)
    mw_results[task_name] = {'success_rate': sr}
    print(f'  SR (ablated): {sr:.1%}')

for h in handles:
    h.remove()

with open(ABLATION_DIR / 'metaworld_ablation.json', 'w') as f:
    json.dump(mw_results, f, indent=2)
print('\n✅ MetaWorld ablation done')


---
## 4. VLABench — `lerobot/pi0` (generalist)

In [ ]:
# ── VLABench Baseline ──────────────────────────────────────────────────────
# lerobot does not have a built-in VLABench env type, so we load the
# VLABench gymnasium environment directly and drive it with lerobot/pi0.
# Note: VLABench/pi0-primitive-10task is in openpi format — not lerobot-compatible.
# Using lerobot/pi0 keeps everything consistent across all three benchmarks.

import sys
sys.path.insert(0, '/content/VLABench')

from VLABench.tasks import TASK_MAP   # registers all VLABench gym envs
import gymnasium as gym

VLA_TASKS   = ['select_toy', 'stack_block', 'put_in_drawer', 'close_drawer',
               'insert_peg', 'press_button', 'turn_tap', 'sweep_to_dustpan',
               'pick_and_place', 'reach_target']
VLA_CKPT    = 'lerobot/pi0'
VLA_EPISODES = 10

policy = load_policy(VLA_CKPT)

def format_obs_vla(obs, device):
    """VLABench obs: dict with image and state keys."""
    batch = {}
    if 'image' in obs:
        img = torch.tensor(obs['image'], dtype=torch.float32).permute(2,0,1) / 255.0
        batch['observation.images.image'] = img.unsqueeze(0).to(device)
    if 'state' in obs:
        batch['observation.state'] = torch.tensor(
            obs['state'], dtype=torch.float32).unsqueeze(0).to(device)
    return batch

vla_results = {}
for task_name in VLA_TASKS:
    print(f'\n📍 Baseline: {task_name}')
    env = gym.make(f'VLABench/{task_name}-v0')
    successes = [run_episode(policy, env, format_obs_vla) for _ in range(VLA_EPISODES)]
    env.close()
    sr = sum(successes) / len(successes)
    vla_results[task_name] = {'success_rate': sr}
    print(f'  SR: {sr:.1%}')

with open(BASELINE_DIR / 'vlabench_baseline.json', 'w') as f:
    json.dump(vla_results, f, indent=2)

overall = sum(r['success_rate'] for r in vla_results.values()) / len(vla_results)
print(f'\n✅ VLABench baseline done  |  Overall SR: {overall:.1%}')


In [ ]:
# ── VLABench Ablation ──────────────────────────────────────────────────────
policy  = load_policy('lerobot/pi0')
handles = hook_state_proj(policy)

vla_ablation = {}
for task_name in VLA_TASKS:
    print(f'\n📍 Ablation: {task_name}')
    env = gym.make(f'VLABench/{task_name}-v0')
    successes = [run_episode(policy, env, format_obs_vla) for _ in range(VLA_EPISODES)]
    env.close()
    sr = sum(successes) / len(successes)
    vla_ablation[task_name] = {'success_rate': sr}
    print(f'  SR (ablated): {sr:.1%}')

for h in handles:
    h.remove()

with open(ABLATION_DIR / 'vlabench_ablation.json', 'w') as f:
    json.dump(vla_ablation, f, indent=2)
print('\n✅ VLABench ablation done')


In [ ]:
# ── Results Summary ───────────────────────────────────────────────────────
import json
from pathlib import Path

def load_json(p):
    p = Path(p)
    return json.loads(p.read_text()) if p.exists() else {}

libero_base   = load_json(BASELINE_DIR / 'libero')    # lerobot-eval output
mw_base       = load_json(BASELINE_DIR / 'metaworld') # lerobot-eval output
vla_base      = load_json(BASELINE_DIR / 'vlabench_baseline.json')
libero_abl    = load_json(ABLATION_DIR / 'libero_ablation.json')
mw_abl        = load_json(ABLATION_DIR / 'metaworld_ablation.json')
vla_abl       = load_json(ABLATION_DIR / 'vlabench_ablation.json')

def mean_sr(d):
    rates = [v['success_rate'] for v in d.values() if 'success_rate' in v]
    return sum(rates)/len(rates) if rates else float('nan')

print('='*55)
print(f'{'Benchmark':<20} {'Baseline SR':>12} {'Ablated SR':>12} {'Drop':>8}')
print('='*55)
for label, base, abl in [('LIBERO', libero_base, libero_abl),
                          ('MetaWorld', mw_base,  mw_abl),
                          ('VLABench', vla_base,  vla_abl)]:
    b, a = mean_sr(base), mean_sr(abl)
    drop = b - a
    print(f'{label:<20} {b:>11.1%} {a:>11.1%} {drop:>7.1%}')
print('='*55)
print('Low drop → state encoder underutilized (paper hypothesis ✅)')


---
# π₀ (3.3B) — Proprioceptive State Utilization Analysis

**Part 2: Standalone Gradient Analysis**  
Uses `lerobot/pi0` (generalist) to measure gradient flow through `state_proj`,
confirming architectural underutilization independent of benchmark-specific fine-tuning.

# π0 (3.3B) - Proprioceptive State Utilization Analysis

**Model**: Physical-Intelligence π0.5-DROID (3.3B parameters)  
**State Encoder**: `state_proj` - **Single Linear layer** (NOT multi-layer MLP)

## ⚠️ Architecture Correction
**Previous (Incorrect) Assumption**: Multipler encoder with separate layers  
**Actual Architecture** (from [openpi repo](https://github.com/Physical-Intelligence/openpi)):
- Vision: PaliGemma ViT encoder
- Language: PaliGemma Gemma model  
- **State**: Single Linear projection (`state_proj`)
- Expert: Separate Expert Gemma for flow matching
- Action: Diffusion-based via flow matching

## Hypothesis
**Proprioceptive state information is underutilized** - despite having a dedicated projection layer, the model may not meaningfully use robot state for action prediction.

## Experimental Design  
1. **Baseline**: Run normal state through state_proj → capture gradients
2. **Ablation**: Zero out robot_state → capture gradients  
3. **Compare**: Calculate gradient change percentage
4. **Verdict**:
   - <10% change = ❌ UNDERUTILIZED (state doesn't matter)
   - 10-30% change = ⚠️ PARTIALLY UTILIZED
   - >30% change = ✅ WELL UTILIZED (model relies on state)

## Key Metrics
- **Gradient norm on state_proj layer**
- **Gradient change % when state is ablated**

---

## 1. Environment Setup

In [1]:
# Check environment
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🚀 Running on Google Colab")
    !nvidia-smi
else:
    print("💻 Running locally")

🚀 Running on Google Colab
Sun Feb 15 04:19:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   49C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------

In [2]:
# On Colab, torch/numpy/matplotlib/pillow/sklearn are pre-installed.
# lerobot will handle its own dependencies cleanly.
# Only pre-install packages that might be missing:
!pip install -q einops imageio

print("✅ Extra prerequisites ready")


✅ Dependencies installed


### π0 (openpi) Specific Setup

**Official Repository**: `Physical-Intelligence/openpi`

**Architecture** (π0.5-DROID, 3.3B params):
- Vision: PaliGemma ViT encoder
- State: `state_proj` - Single Linear layer (NOT multi-layer encoder)
- Language: PaliGemma language model  
- Expert: Separate Gemma model for flow matching
- Action: Flow matching + diffusion-based

**Critical Requirements**:
```bash
# Install uv (modern Python package manager)
curl -LsSf https://astral.sh/uv/install.sh | sh

# OR using pip
pip install uv

# Install transformers with required patches
pip install transformers==4.53.2

# Clone openpi repository
git clone https://github.com/Physical-Intelligence/openpi.git
cd openpi
uv pip install -e .
```

**Model Loading**:
```python
from openpi.inference.policy import policy_config

# Load π0.5-DROID model
policy = policy_config.create_trained_policy(
    ckpt_path="droid-dataset",
    ckpt_step="latest"  
)
model = policy.model
```

**Note**: π0 uses **single Linear `state_proj` layer**, not a multi-layer encoder. See updated hooks for correct architecture.

In [ ]:
# ── Part 2: lerobot install (skip if already installed from Part 1) ─────────
import importlib, subprocess, sys

if importlib.util.find_spec('lerobot') is None:
    print("📦 Installing lerobot...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lerobot"], check=True)
    print("✅ lerobot installed")
else:
    print("✅ lerobot already installed (from Part 1)")

# Verify state_proj is in the implementation
from lerobot.policies.pi0.modeling_pi0 import Pi0Policy
import inspect
assert 'state_proj' in inspect.getsource(Pi0Policy), 'state_proj not found!'
print("✅ state_proj confirmed in Pi0Policy")
print("\n✅ π0 environment ready (lerobot backend)")

In [ ]:
# ── Clone repo (skip if already cloned from Part 1) ─────────────────────────
import os, sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    if not os.path.exists('/content/EmbodiedLLM'):
        import subprocess
        subprocess.run(['git', 'clone', 'https://github.com/PrithviTM-glitch/EmbodiedLLM.git',
                        '/content/EmbodiedLLM'], check=True)
        subprocess.run(['git', 'checkout', 'AnalyseMultipleHooks'],
                       cwd='/content/EmbodiedLLM', check=True)
    else:
        print("✅ EmbodiedLLM already cloned (from Part 1)")

    # Change to MultipleHooksStudy dir and add hooks to path
    hooks_dir = '/content/EmbodiedLLM/MultipleHooksStudy'
    os.chdir(hooks_dir)
    if hooks_dir not in sys.path:
        sys.path.insert(0, hooks_dir)
    print(f"✅ Working directory: {os.getcwd()}")
else:
    if not os.path.exists('hooks'):
        print("⚠️ Please run from MultipleHooksStudy directory")
    else:
        print("✅ Hook files found")

## 2. Import Hook Framework

In [4]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Add hooks to path
sys.path.insert(0, str(Path.cwd() / 'hooks'))

from hooks.model_specific.pi0_hooks import Pi0Hooks

print("✅ π0 hook framework imported")

✅ π0 hook framework imported


## 3. Load π0 Model

**Official Loading Method** (Recommended):
```python
# Use openpi's official policy loading
sys.path.insert(0, 'openpi')
from openpi.inference.policy import policy_config

# Load π0.5-DROID (3.3B) with PyTorch support
policy = policy_config.create_trained_policy(
    ckpt_path="droid-dataset",  # or your checkpoint path
    ckpt_step="latest"
)
model = policy.model
```

**Corrected Architecture**:
- `paligemma`: PaliGemma model (vision + language encoder)
  - `model.vision_encoder`: ViT for image processing
  - `model.language_model`: Gemma for text processing
- `gemma_expert`: Separate Expert Gemma for flow matching
- `state_proj`: **Single Linear layer** (NOT multi-layer MLP!)
  - Maps robot state to embedding space
- `action_in_proj`, `action_out_proj`: Linear layers for action processing

**Previous Assumption (INCORRECT)**:
- ❌ Multi-layer `proprio_encoder` - **This doesn't exist!**
- ❌ Causal attention layers - Handled by standard transformer

**Actual Implementation**:
- ✅ Single Linear `state_proj` layer for state encoding
- ✅ PaliGemma handles vision + language fusion
- ✅ Expert Gemma performs flow matching for actions

**Note**: See updated `pi0_hooks.py` for corrected hook attachment targeting `state_proj`.

In [ ]:
import os, sys, torch
from lerobot.policies.pi0.modeling_pi0 import Pi0Policy

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ── Load lerobot/pi0 from HuggingFace ──────────────────────────────────
# lerobot/pi0 = original π0 architecture (4 B params):
#   • PaliGemma vision-language backbone
#   • state_proj: nn.Linear(max_state_dim, expert_width)   ← what we study
#   • Expert Gemma for flow-matching action prediction
# This is NOT π0.5 (which has no state encoder at all).
print("\nDownloading lerobot/pi0 from HuggingFace (first run ~15 GB, cached after)...")
policy = Pi0Policy.from_pretrained("lerobot/pi0")
policy = policy.to(device)
print("✅ lerobot/pi0 loaded")

# ── Locate the underlying PI0Pytorch model with state_proj ──────────────
model = None
for attr in ["model_pi0", "pi0", "model", "policy_model", "pi0_model"]:
    candidate = getattr(policy, attr, None)
    if candidate is not None and hasattr(candidate, "state_proj"):
        model = candidate
        print(f"✅ PI0Pytorch found at policy.{attr}")
        break

if model is None:
    # Fallback: search all Module children
    for name, mod in policy.named_modules():
        if hasattr(mod, "state_proj") and isinstance(mod.state_proj, torch.nn.Linear):
            model = mod
            print(f"✅ PI0Pytorch found via named_modules: {name}")
            break

if model is None:
    raise RuntimeError(
        "Could not find PI0Pytorch with state_proj. "
        "Check lerobot version or inspect policy.__dict__"
    )

# ── Confirm state_proj ──────────────────────────────────────────────────
sp = model.state_proj
print(f"\n✅ state_proj confirmed: nn.Linear({sp.in_features}, {sp.out_features})")
print(f"   Input: {sp.in_features}-dim padded robot state")
print(f"   Output: {sp.out_features}-dim expert embedding")

cfg = policy.config
print(f"\nPolicy config:")
print(f"  max_state_dim : {cfg.max_state_dim}")
print(f"  action_dim    : {cfg.action_dim}")
print(f"  chunk_size    : {cfg.chunk_size}")

try:
    n_params = sum(p.numel() for p in model.parameters()) / 1e9
    print(f"\nModel parameters: {n_params:.2f}B")
except:
    pass


Using device: cuda
Loading real π0 model from Physical Intelligence...
Installing openpi package...
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 4.3 MB/s eta 0:00:00


## 4. Discover Model Structure

**Critical**: Verify separate multi-layer proprio encoder!

In [ ]:
# Initialize hook manager and discover model structure
hook_manager = Pi0Hooks(model)
structure = hook_manager.discover_model_structure()

print("\n" + "="*60)
print("π0 Model Structure Discovery")
print("="*60)
print(f"Model Name: {structure['model_name']}")
print(f"State Encoder Type: {structure.get('proprio_encoder_type', 'unknown')}")

print(f"\nComponents Found:")
for component, attr in structure['components'].items():
    print(f"  - {component}: {attr}")

# π0 uses a single Linear state_proj (not multi-layer) — this is expected
if 'state_proj' in structure['components']:
    sp = model.state_proj
    print(f"\n✅ state_proj confirmed: nn.Linear({sp.in_features} → {sp.out_features})")
    print(f"   π0 encodes robot state via a single linear projection (expected architecture)")
else:
    print("\n⚠️ state_proj not found in discovered components — check model structure")

print("="*60)

## 5. Attach Analysis Hooks

In [ ]:
# Attach gradient hooks only (focus on proprio_encoder)
print("Attaching gradient hooks to proprio_encoder layers...")
hook_manager.attach_gradient_hooks()
print("✓ Gradient hooks attached to multi-layer proprio_encoder")
print("\nWill compare gradients: Normal state vs Zero state")

## 6. Prepare Sample Data

In [ ]:
# lerobot uses a standardised dict-based batch format.
# Keys follow the pattern "observation.<sensor>" and "action".
# ─────────────────────────────────────────────────────────────────────

cfg = policy.config
batch_size   = 1
max_state_dim = cfg.max_state_dim   # e.g. 32
action_dim    = cfg.action_dim      # e.g. 32
chunk_size    = cfg.chunk_size      # e.g. 50

# Synthetic batch — random values are fine for gradient analysis.
# The gradient pattern reflects the model's *architectural* reliance
# on state_proj, independent of task-specific data distribution.
batch = {
    # Proprioceptive state (padded to max_state_dim)
    "observation.state": torch.randn(batch_size, max_state_dim, device=device),
    # Camera image (224×224 RGB, single front camera)
    "observation.images.front": torch.zeros(batch_size, 3, 224, 224, device=device),
    # Ground-truth actions for flow-matching loss
    "action": torch.randn(batch_size, chunk_size, action_dim, device=device),
}

print("✅ Synthetic batch prepared for π0 gradient analysis")
for k, v in batch.items():
    print(f"  {k}: {v.shape}")
print("\nNote: Synthetic data → gradient magnitudes reflect architectural utilisation,")
print("       not task-specific learned correlations.")


## 7. Run Forward and Backward Pass

In [ ]:
# lerobot policy.forward(batch) → (loss, output_dict)
# loss is the flow-matching MSE loss used during Pi0 training.
policy.train()
policy.zero_grad()

print("Running π0 forward pass (flow-matching loss)...")
try:
    loss, output_dict = policy.forward(batch)
    loss = loss.mean() if loss.dim() > 0 else loss

    print("Running backward pass...")
    loss.backward()

    print(f"✅ Forward + backward completed")
    print(f"   Flow-matching loss: {loss.item():.4f}")
    if output_dict:
        print(f"   Output keys: {list(output_dict.keys())}")

except TypeError:
    # Older lerobot versions return only loss (no output_dict)
    loss = policy.forward(batch)
    if isinstance(loss, (tuple, list)):
        loss = loss[0]
    loss = loss.mean() if loss.dim() > 0 else loss
    loss.backward()
    print(f"✅ Forward + backward (single-return variant). Loss: {loss.item():.4f}")

except Exception as e:
    print(f"⚠️  policy.forward failed: {e}")
    print("   Trying direct PI0Pytorch model.forward...")
    # Build SimpleNamespace observation if needed
    from types import SimpleNamespace
    obs = SimpleNamespace(
        images=batch["observation.images.front"].unsqueeze(1),   # (B,1,C,H,W)
        state=batch["observation.state"],
        tokenized_prompt=torch.zeros(batch_size, 20, dtype=torch.long, device=device),
        tokenized_prompt_mask=torch.ones(batch_size, 20, dtype=torch.bool, device=device),
    )
    actions = batch["action"]
    loss = model(obs, actions).mean()
    loss.backward()
    print(f"✅ Direct model.forward succeeded. Loss: {loss.item():.4f}")


## 8. Baseline: Capture Gradients with Normal State

In [ ]:
# ── Baseline: capture state_proj weight gradient after normal forward+backward ──
# The weight gradient tells us how strongly the loss signal propagates through
# state_proj, i.e. how much the network was relying on the state input.

grad = model.state_proj.weight.grad
if grad is not None:
    baseline_norm = grad.norm().item()
    print(f"✅ state_proj weight gradient norm (normal state): {baseline_norm:.6f}")
    print(f"   Weight shape: {model.state_proj.weight.shape}")
    print(f"   This is our baseline — model sees real robot state")
else:
    baseline_norm = 0.0
    print("⚠️ No gradient on state_proj.weight — did the forward+backward pass run?")
    print("   Re-run cell 7 before this cell.")

## 9. Ablation: Zero Out State Encoder Output

**Better Ablation Strategy**: Zero the OUTPUT of `proprio_encoder`, not the input. 

This directly tests: "What if the multi-layer encoder contributed nothing?"

In [ ]:
# ── Ablation: zero state_proj output, re-run forward+backward ─────────────────
# Attaching a forward hook that returns zeros for state_proj output means the
# downstream expert model receives no proprioceptive state information at all.
# We then check how much the weight gradient changes vs the baseline.

def zero_output_hook(module, input, output):
    """Replace state_proj output with zeros — model acts as if state is absent."""
    return torch.zeros_like(output)

ablation_handle = model.state_proj.register_forward_hook(zero_output_hook)
print(f"✓ Ablation hook on state_proj "
      f"({model.state_proj.in_features} → {model.state_proj.out_features})")
print(f"  Hook returns zeros → model acts as if state is absent")

# Re-run forward + backward with hook active
policy.zero_grad()
print("\nRunning ablation forward+backward (state_proj output = zeros)...")
try:
    loss_ablated, _ = policy.forward(batch)
    loss_ablated = loss_ablated.mean() if loss_ablated.dim() > 0 else loss_ablated
except TypeError:
    loss_ablated = policy.forward(batch)
    if isinstance(loss_ablated, (tuple, list)):
        loss_ablated = loss_ablated[0]
    loss_ablated = loss_ablated.mean() if loss_ablated.dim() > 0 else loss_ablated
loss_ablated.backward()

# Remove hook immediately after backward
ablation_handle.remove()
print("✓ Ablation complete — hook removed")

## 10. Compare Gradients: Normal vs Ablation

In [ ]:
# ── Compare: ablated gradient norm vs baseline ─────────────────────────────────
grad_ablated = model.state_proj.weight.grad
if grad_ablated is not None:
    ablated_norm = grad_ablated.norm().item()
else:
    ablated_norm = 0.0

grad_change_pct = (abs(baseline_norm - ablated_norm) / baseline_norm * 100
                   if baseline_norm > 0 else 0.0)

print(f"{'='*60}")
print(f"GRADIENT COMPARISON")
print(f"{'='*60}")
print(f"Normal state   → state_proj ‖∇w‖: {baseline_norm:.6f}")
print(f"Ablated state  → state_proj ‖∇w‖: {ablated_norm:.6f}")
print(f"Change:                            {grad_change_pct:.1f}%")
print(f"{'='*60}")
print(f"\nLoss with normal state:   {loss.item():.4f}")
print(f"Loss with ablated state:  {loss_ablated.item():.4f}")
print(f"Loss Δ: {abs(loss.item() - loss_ablated.item()):.4f}")

## 11. Verdict: Is Proprioceptive State Utilized?

In [ ]:
# ── Verdict ────────────────────────────────────────────────────────────────────
if grad_change_pct < 10:
    verdict = "UNDERUTILIZED"
    explanation = ("When state_proj output is zeroed, weight gradients barely change (<10%). "
                   "The model does not rely meaningfully on the proprioceptive state encoder.")
elif grad_change_pct < 30:
    verdict = "PARTIALLY UTILIZED"
    explanation = ("Some gradient sensitivity to state removal (10–30%). "
                   "The dependency exists but is weak.")
else:
    verdict = "WELL UTILIZED"
    explanation = ("Strong gradient response (>30%) when state_proj output is ablated. "
                   "The model genuinely uses the proprioceptive state input.")

print(f"\n{'='*60}")
print(f"VERDICT: {verdict}")
print(f"{'='*60}")
print(f"\nGradient change: {grad_change_pct:.1f}%")
print(f"\n{explanation}")
print(f"\nMethodology:")
print(f"  • Model: lerobot/pi0 (generalist, {sum(p.numel() for p in model.parameters())/1e9:.1f}B params)")
print(f"  • Layer: state_proj  nn.Linear({model.state_proj.in_features}, {model.state_proj.out_features})")
print(f"  • Loss:  flow-matching MSE (same objective as Pi0 training)")
print(f"  • Metric: ‖∇w‖ of state_proj.weight — normal vs output-ablated")
print(f"{'='*60}")

# ── Save to Google Drive (if available) ────────────────────────────────────────
import json, os
results = {
    'model': 'lerobot/pi0 (generalist)',
    'state_encoder': 'state_proj  nn.Linear',
    'state_proj_in': model.state_proj.in_features,
    'state_proj_out': model.state_proj.out_features,
    'ablation_method': 'zero_state_proj_output',
    'baseline_grad_norm': float(baseline_norm),
    'ablated_grad_norm': float(ablated_norm),
    'gradient_change_pct': float(grad_change_pct),
    'baseline_loss': float(loss.item()),
    'ablated_loss': float(loss_ablated.item()),
    'verdict': verdict,
}

# Save locally
local_path = 'pi0_gradient_results.json'
with open(local_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f"\n✓ Results saved locally: {local_path}")

# Save to Drive if mounted
drive_gradient_dir = '/content/drive/MyDrive/pi0_study/Results/gradient'
if os.path.exists('/content/drive/MyDrive'):
    os.makedirs(drive_gradient_dir, exist_ok=True)
    drive_path = f'{drive_gradient_dir}/pi0_gradient_results.json'
    with open(drive_path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"✓ Results saved to Drive: {drive_path}")